In [1]:
%load_ext autoreload
%autoreload 2


# Import Libraries

In [84]:
import os

import pandas as pd
import nltk
import spacy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from catboost import Pool, CatBoostClassifier


# Read Data

In [7]:
path = "../../"
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
print(f"Number of rows and columns in the train data set: {train.shape}")
print(f"Number of rows and columns in the test data set: {test.shape}")
train.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [29]:
train['rate'].value_counts()

rate
5    26069
4     9922
3     6126
1     4138
2     2410
Name: count, dtype: int64

In [8]:
nlp = spacy.load("ru_core_news_sm")
stopwords = nlp.Defaults.stop_words

In [9]:
train['cleaned_text'] = train['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        not token.is_stop
        and not token.is_punct
        and not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [10]:
vec = TfidfVectorizer(max_features = 100)
bow = vec.fit_transform(train['cleaned_text'])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(bow, train['rate'], shuffle = True, random_state=2025)

In [30]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes = classes, y = y_train)
class_weights=dict(zip(classes,weights))
class_weights

{1: 2.324713375796178,
 2: 3.9975903614457833,
 3: 1.596937212863706,
 4: 0.9847025495750709,
 5: 0.3734192756292204}

In [31]:
model = CatBoostClassifier(loss_function='MultiClass', iterations = 1000, learning_rate = 0.05, class_weights = class_weights,  early_stopping_rounds= 200, random_seed=42)


In [ ]:
model.fit(X_train,y_train,eval_set=(X_test, y_test),verbose=False)

In [33]:
predictions = model.predict(X_test)


In [34]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           1       0.29      0.56      0.38       998
           2       0.13      0.20      0.16       584
           3       0.24      0.33      0.28      1555
           4       0.37      0.33      0.35      2509
           5       0.80      0.62      0.70      6521

    accuracy                           0.50     12167
   macro avg       0.37      0.41      0.37     12167
weighted avg       0.57      0.50      0.52     12167



# Preparing the data and creating Catboost model

In [64]:
X_train, X_test, y_train, y_test = train_test_split(train, train['rate'], shuffle = True, random_state=2025)

In [65]:
X_train.head()

,rate,text,cleaned_text
38373,5,"Хороший выбор товаров, побольше, чем у соседне...",хороший выбор товар большой соседний малый про...
37650,5,"Хорошая пятёрочка, всегда всё на своих местах....",хороший место небольшой парковка
11122,3,Последнее время очень упал общий уровень магаз...,последний время упасть общий уровень магазин с...
1978,5,"Хороший магазин! Отличное местоположение, ассо...",хороший магазин отличный местоположение ассорт...
42401,5,Обожаю кофе в этом магазине,обожать кофе магазин


In [74]:
X_train = X_train.drop(['rate','text'],axis = 1)
X_test = X_test.drop(['rate','text'],axis = 1)

In [81]:
model = CatBoostClassifier( iterations = 1000, class_weights = class_weights,  early_stopping_rounds= 200, random_seed=42)


In [79]:
model.fit(X_train, y_train, text_features=[0],verbose=False,eval_set=(X_test, y_test) )

In [80]:
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           1       0.48      0.62      0.54       998
           2       0.17      0.24      0.20       584
           3       0.33      0.38      0.35      1555
           4       0.40      0.41      0.40      2509
           5       0.84      0.73      0.78      6521

    accuracy                           0.58     12167
   macro avg       0.44      0.48      0.45     12167
weighted avg       0.62      0.58      0.60     12167



Оптимизация гиперпараметров

In [88]:
params = {
    'text_features' : [[0]],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [300, 500, 700],
    'l2_leaf_reg': [1, 3, 5]
}

In [90]:
grid_search = GridSearchCV(model, params, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)

KeyboardInterrupt: 

In [91]:
X_train = train["cleaned_text"]
y_train = train["rate"]

X_test = test["text"]


model = CatBoostClassifier(
    iterations=100,
    depth=5,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    text_features=[0],
    verbose=False
)

# Predict

In [6]:
# Preparing data in Pool format
dataset_test = Pool(
    data=X_test,
    text_features=[0]
)
predict_classes = model.predict(dataset_test)
predictions = predict_classes

# Create submission

In [7]:
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
sample_submission["rate"] = predictions
sample_submission.head()

,rate
0,5
1,5
2,5
3,4
4,5


In [8]:
sample_submission.to_csv("submission.csv", index=False)